# 판촉사랑 납품사례 크롤링 인수인계용 노트북

이 노트북은 판촉사랑 납품사례 페이지에서 상품 정보를 수집해 엑셀로 저장하는 작업용 파일입니다.

## 전체 작업 흐름

1. **기본 설정 확인**
   - 어디서부터 어디까지 크롤링할지 페이지 범위를 정합니다.
   - 결과 파일명, 중간 저장 파일명, 오류 로그 파일명을 확인합니다.

2. **1단계 목록 수집**
   - 납품사례 목록 페이지에서 기본 정보만 먼저 수집합니다.
   - 수집 항목: 상품코드, 구매처분류(중/소/세), 상품명, 인쇄방법, 업종/행사, 등록일, 상세URL, 납품사례ID, 수집키입니다.
   - 저장 파일: `list_checkpoint.csv`입니다.

3. **2단계 상세 수집**
   - 1단계에서 저장한 상세URL을 하나씩 열어서 상세 정보를 수집합니다.
   - 수집 항목: 상품분류(대/중/소), 최소인쇄수량, 최소주문수량, 가격 정보입니다.
   - 상세 수집은 Selenium 브라우저를 사용합니다.

4. **엑셀 저장**
   - 중간 저장 파일인 `detail_checkpoint.csv`와 확인용 `detail_checkpoint.xlsx`를 기준으로 결과 엑셀을 만듭니다.
   - 최종 저장 파일: `crawl_result.xlsx`입니다.
   - 이 최종 엑셀 안에 `납품사례ID`, `수집키`가 함께 저장됩니다.

5. **납품사례ID / 수집키 포함 저장**
   - 구매처분류(중/소/세)는 크롤링 중 `업종/행사` 값에서 자동 생성되어 결과 엑셀에 포함됩니다.
   - `crawl_result.xlsx` 파일 자체에 `납품사례ID`, `수집키`가 포함됩니다.
   - 후처리 셀은 기존 결과 파일 보정용으로만 사용하면 됩니다.

## 실행 순서

아래 순서대로 실행하면 됩니다.

1. **라이브러리 불러오기 셀 실행**
2. **기본 설정 셀 실행**
3. **공통 함수 셀 실행**
4. **목록 페이지 파싱 함수 셀 실행**
5. **상세 페이지 파싱 함수 셀 실행**
6. **엑셀 저장 함수 셀 실행**
7. **1단계/2단계 수집 함수 셀 실행**
8. **1단계 실행 셀 실행**: 목록 정보와 상세URL 수집
9. **2단계 테스트 셀 실행**: 상세URL 1건이 정상 파싱되는지 확인
10. **2단계 실행 셀 실행**: 전체 상세페이지 수집
11. **수동 저장 셀 실행**: 필요 시 최종 엑셀 저장
12. **상태 확인 셀 실행**: 체크포인트와 오류 로그 확인
13. **납품사례ID/수집키 추가 셀 실행**

## 중간에 멈췄을 때

- `list_checkpoint.csv`가 있으면 목록 수집은 이어서 할 수 있습니다.
- `detail_checkpoint.csv`와 확인용 `detail_checkpoint.xlsx`가 있으면 상세 수집은 이어서 할 수 있습니다.
- 오류 내용은 `error_log.csv`에서 확인합니다.
- 이미 완료된 수집키는 다시 수집하지 않도록 설계되어 있습니다.

## 주의사항

- 크롤링 속도를 너무 빠르게 하면 사이트에서 400, 403, 429 오류가 발생할 수 있습니다.
- 상세페이지 수집은 Chrome 창이 열릴 수 있습니다. 크롬 창이 뜨더라도 정상 동작입니다.
- 작업 전 같은 폴더에 `data/supply_case_template.xlsx` 템플릿 파일이 있는지 확인해야 합니다.
- 최종 엑셀 저장 전까지는 체크포인트 CSV가 가장 중요한 중간 저장 파일입니다.


# 판촉사랑 납품사례 크롤링 v11

## 2단계 분리 수집 버전

이 버전은 수집을 두 단계로 분리했습니다.

1. **목록 수집**
   - 상품코드
   - 상품명
   - 인쇄방법
   - 업종/행사
   - 구매처분류(중), 구매처분류(소), 구매처분류(세)
   - 등록일
   - 상품명에 걸려 있는 상세페이지 링크

2. **상세 수집**
   - 상품분류(대/중/소)
   - 최소인쇄수량
   - 최소주문수량
   - 대량가격 / 중간가격 / 소량가격

상세페이지에서 400 오류가 발생해도 목록 데이터는 먼저 안전하게 확보할 수 있습니다.

In [ ]:
# 필요 라이브러리
# 처음 실행 시 아래 설치가 필요할 수 있습니다.
# !pip install requests beautifulsoup4 pandas openpyxl lxml selenium webdriver-manager

import re
import time
import random
import html
import os
import traceback
import shutil
from copy import copy
from pathlib import Path
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, urlunparse

import requests
import pandas as pd
from bs4 import BeautifulSoup

from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager


## 1단계. 기본 설정 확인하기

이 셀은 크롤링을 시작하기 전에 작업 조건을 정하는 곳입니다.

여기서 주로 확인할 값은 아래입니다.

- `START_PAGE`: 몇 페이지부터 수집할지 정합니다.
- `END_PAGE`: 몇 페이지까지 수집할지 정합니다.
- `TEMPLATE_XLSX`: 결과 엑셀의 기준 양식 파일입니다.
- `OUTPUT_XLSX`: 최종 결과 엑셀 파일명입니다.
- `CHECKPOINT_CSV`: 상세 수집 중간 저장 파일입니다.
- `ERROR_LOG_CSV`: 오류가 난 항목을 기록하는 파일입니다.
- `LIST_ONLY_CSV`: 1단계 목록 수집 결과 파일입니다.

실행 전에는 보통 `START_PAGE`, `END_PAGE`, 저장 파일명만 확인하면 됩니다.
크롤링이 자주 막히면 `REQUEST_SLEEP_MIN`, `REQUEST_SLEEP_MAX` 값을 늘려서 요청 간격을 더 길게 잡습니다.


In [ ]:

# =========================
# 1-1 단계 기본 설정
# =========================


BASE_URL = "https://www.87sarang.com"

START_URL = (
    "https://www.87sarang.com/customercenter/supplycaselist.asp?"
    "page=1&lcode=&mcode=&scode=&iscolor=&isSamllQty=&origin=&isOrderMade="
    "&freed=&freep=&orderby=12&orderbytype=desc&printtype=&price01=&price02="
    "&mcd=&stype=N&srch="
)
#START_PAGE = 크롤링 시작 페이지
#END_PAGE = 크롤링 끝나는 페이지

START_PAGE = 1
END_PAGE = 9999


# =========================
# 1-2 단계 저장 파일 설정
# 파일명 변경 가능

#TEMPLATE_XLSX	결과 엑셀의 기본 양식 파일
#OUTPUT_XLSX	최종 크롤링 결과 엑셀
#CHECKPOINT_CSV	중간 저장 파일, 재시작할 때 이어서 하기 위함
#ERROR_LOG_CSV	오류 발생 내역 저장
#LIST_ONLY_CSV	목록 페이지만 먼저 수집한 결과 저장
# =========================
BASE_DIR = Path.cwd()
SAVE_DIR = BASE_DIR / "output"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

TEMPLATE_XLSX = "data/supply_case_template.xlsx"

# 기본 저장 파일: 현재 폴더에 저장
OUTPUT_XLSX = "crawl_result.xlsx"

CHECKPOINT_CSV = "detail_checkpoint.csv"
CHECKPOINT_XLSX = "detail_checkpoint.xlsx"
ERROR_LOG_CSV = "error_log.csv"
LIST_ONLY_CSV = "list_checkpoint.csv"

# 복사 저장 위치: 현재 폴더 하위 output 폴더
OUTPUT_XLSX_DIR = SAVE_DIR / Path(OUTPUT_XLSX).name
CHECKPOINT_CSV_DIR = SAVE_DIR / Path(CHECKPOINT_CSV).name
CHECKPOINT_XLSX_DIR = SAVE_DIR / Path(CHECKPOINT_XLSX).name
ERROR_LOG_CSV_DIR = SAVE_DIR / Path(ERROR_LOG_CSV).name
LIST_ONLY_CSV_DIR = SAVE_DIR / Path(LIST_ONLY_CSV).name

def sync_to_dir(path):
    """
    현재 폴더에 저장된 결과 파일을 output 폴더에도 같은 이름으로 복사합니다.
    """
    src = Path(path)
    if not src.exists():
        return

    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    dst = SAVE_DIR / src.name

    # 원본과 대상이 같은 경우는 복사하지 않음
    try:
        if src.resolve() == dst.resolve():
            return
    except FileNotFoundError:
        pass

    shutil.copy2(src, dst)

print("현재 폴더:", BASE_DIR)
print("복사 저장 폴더:", SAVE_DIR)

# =========================
# 1-3 단계 저장 주기 설정
# =========================
SAVE_EXCEL_EVERY_ROWS = 5000


# =========================
# 1-4 단계. 요청 간격 설정

# 최소 5초 ~ 최대 12초 사이 랜덤 대기
# =========================

REQUEST_SLEEP_MIN = 5.0
REQUEST_SLEEP_MAX = 12.0

# =========================
# 1-5 단계. 재시도 설정
# =========================

MAX_RETRIES = 2
RETRY_SLEEP = 15

# =========================
# 1-6 단계. 요청 헤더 설정
# =========================

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/136.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.87sarang.com/",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

session = requests.Session()
session.headers.update(HEADERS)

# =========================
# 2 단계 분리 수집 설정
# =========================

BUYER_CATEGORY_COLUMNS = ["구매처분류(중)", "구매처분류(소)", "구매처분류(세)"]

LIST_COLUMNS = [
    "수집키", "납품사례ID", "수집페이지",
    "상품코드", *BUYER_CATEGORY_COLUMNS, "상품명",
    "인쇄방법", "업종/행사", "등록일", "상세URL",
    "목록수집상태"
]

# 페이지 오류 400 / 403 / 429 응답 발생 시 추가 대기 시간
# 실제 대기시간 = BLOCK_STATUS_SLEEP * 재시도횟수
BLOCK_STATUS_SLEEP = 60

# 상세페이지 요청 전에 목록페이지를 한 번 열어서 Referer / 세션 흐름을 맞출지 여부
WARMUP_REFERER_BEFORE_DETAIL = True

# 상세페이지가 끝까지 열리지 않아도 목록에서 가져온 기본정보는 엑셀에 저장할지 여부
DETAIL_ERROR_SAVE_AS_ROW = True

# 기존 체크포인트에 '상세페이지 오류'로 저장된 상품을 다시 시도할지 여부
# True로 바꾸면 오류 행을 체크포인트에서 제거하고 다시 상세 수집합니다.
RETRY_DETAIL_ERRORS = False

# 일정 건수마다 긴 휴식 시간을 줘서 400 오류 가능성을 줄임
COOLDOWN_EVERY_ITEMS = 999999
COOLDOWN_SLEEP_MIN = 0
COOLDOWN_SLEEP_MAX = 0


# =========================
# 3단계 상세페이지 수집 방식
# =========================
# list 단계: 기존 requests + BeautifulSoup
# detail 단계: 기본 Selenium 사용
# requests: 빠르지만 상세페이지에서 400 오류가 날 수 있음
# selenium: 느리지만 실제 브라우저로 열기 때문에 400 오류가 줄어듦
DETAIL_FETCH_MODE = "selenium"

# Selenium 브라우저 표시 여부
# False: 크롬 창이 보임
# True: 창 없이 실행. 단, 사이트에 따라 headless에서 다르게 동작할 수 있음
SELENIUM_HEADLESS = False

# Selenium 페이지 로딩 대기
SELENIUM_WAIT_SECONDS = 5.0
SELENIUM_PAGELOAD_TIMEOUT = 20

# Selenium 상세페이지 요청 간격
SELENIUM_SLEEP_MIN = 0.4
SELENIUM_SLEEP_MAX = 1.2

# 일정 건수마다 브라우저 재시작. 장시간 실행 안정성용
# Selenium 창이 자동으로 닫았다가 다시 열림
# ex): 500개 저장이 되면 다시 닫았다가 열림
SELENIUM_RESTART_EVERY = 500


# =========================
# 4단계 직접 URL 수집 설정
# =========================
# True: 납품사례 목록주소를 먼저 열지 않고, LIST_ONLY_CSV의 상세URL(I열)로 바로 접속
DETAIL_DIRECT_URL_ONLY = True

# 기존 체크포인트에서 상세 오류 또는 상세값이 비어 있는 행을 다시 시도할지 여부
RETRY_BLANK_DETAIL_ROWS = True


# =========================
# Selenium 빠르게추출
# =========================
# True: 이미지 로딩을 막고, 페이지 전체 로딩 완료를 기다리지 않음
# False: 이미지 로딩이 되어, 페이지 로딩 시간이 있음
# HTML 텍스트 기반 추출이므로 일반적으로 True 권장
FAST_SELENIUM_MODE = True

# 상세페이지 핵심 텍스트가 나오면 고정 대기시간을 다 기다리지 않고 바로 파싱
DETAIL_READY_MARKERS = [
    "현재위치",
    "즉시 할인가",
    "즉시할인가",
    "일반 판매가",
    "일반판매가",
    "최소인쇄수량",
    "최소주문수량",
    "가격문의",
]


## 실행 방법

1. 상단 기본 설정에서 `START_PAGE`, `END_PAGE`를 확인합니다.
2. 먼저 **1단계 실행 셀**의 `collect_list_only()`를 실행합니다.
   - 목록페이지에서 `상품코드`, `상품명`, `인쇄방법`, `업종/행사`, `등록일`, `상세URL`만 저장합니다.
   - 저장 파일: `87sarang_supplycase_list_only.csv`
3. 그다음 **2단계 실행 셀**의 `run_detail_from_list()`를 실행합니다.
   - 저장된 상세URL에 하나씩 접속해서 `상품분류`, `최소수량`, `가격`을 수집합니다.
   - 최종 저장 파일: `crawl_result.xlsx`

### 장점

- 상세페이지 400 오류가 나도 목록 데이터는 먼저 확보됩니다.
- 상세 수집만 나중에 다시 실행할 수 있습니다.
- 기존 체크포인트 기준으로 이미 상세 수집된 상품은 건너뜁니다.
- 상세페이지 오류 상품도 기본정보는 엑셀에 남깁니다.

### 상세 오류 상품 다시 시도

기존에 `상세페이지 오류`로 저장된 상품을 다시 시도하려면 기본 설정에서 아래 값을 바꾼 뒤 2단계를 실행합니다.

```python
RETRY_DETAIL_ERRORS = True
```

## 2단계. 공통 유틸 함수와 Selenium 브라우저 함수 준비

이 셀은 여러 곳에서 반복해서 쓰는 기능을 함수로 만들어둔 부분입니다.

주요 역할은 아래입니다.

- 저장할 컬럼명 정의
- 페이지 URL 만들기
- 웹페이지 요청하기
- 가격 텍스트를 숫자로 정리하기
- 상품분류 HTML을 텍스트로 정리하기
- 중복 방지를 위한 수집키 만들기
- 체크포인트 CSV 불러오기
- 오류 로그 저장하기
- Selenium 크롬 브라우저 시작/재시작/종료하기
- Selenium으로 상세페이지 HTML 가져오기

이 셀은 직접 결과를 만드는 셀이 아니라, 뒤쪽 실행 셀들이 사용할 “도구 상자”를 준비하는 셀입니다.
반드시 실행해야 뒤쪽 크롤링 셀이 정상 작동합니다.


In [ ]:
# =========================
# 공통 유틸 함수
# =========================


#저장할 컬럼 정의
# 구매처분류(중/소/세)는 업종/행사 값을 ">" 기준으로 나누어 크롤링 중 바로 생성합니다.
COLUMNS = [
    "No", "상품코드", *BUYER_CATEGORY_COLUMNS, "상품명",
    "상품분류(대)", "상품분류(중)", "상품분류(소)",
    "최소인쇄수량", "최소주문수량",
    "대량가격(원)", "중간가격(원)", "소량가격(원)",
    "인쇄방법", "업종/행사", "등록일",
    "납품사례ID", "수집키"
]
#컬럼	                    의미
#======================================================
#수집키	                중복 방지용 고유값, 최종 엑셀 포함
#납품사례ID	             납품사례 고유 ID, 최종 엑셀 포함
#수집페이지	            몇 페이지에서 가져왔는지
#상세URL	            상세페이지 주소
#수집상태	        완료 / 오류 / 상세페이지 오류 등

INTERNAL_COLUMNS = [
    "수집페이지", "상세URL", "수집상태"
]

ALL_SAVE_COLUMNS = COLUMNS + INTERNAL_COLUMNS


def sleep_random():
    time.sleep(random.uniform(REQUEST_SLEEP_MIN, REQUEST_SLEEP_MAX))

    
#목록 페이지 URL에서 page= 값만 바꿔주는 함수

def build_page_url(base_url: str, page: int) -> str:
    parsed = urlparse(base_url)
    qs = parse_qs(parsed.query, keep_blank_values=True)
    qs["page"] = [str(page)]
    new_query = urlencode(qs, doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def fetch_soup(url: str, timeout: int = 20, referer: str = None) -> BeautifulSoup:
    """
    URL을 요청해서 BeautifulSoup 객체를 반환합니다.

    referer가 있으면 목록페이지에서 상세페이지로 이동하는 것처럼
    Referer 헤더를 함께 전달합니다.

    400/403/429가 나오면 바로 빠르게 재시도하지 않고
    목록페이지를 한 번 열어 세션 흐름을 맞춘 뒤 대기 후 재시도합니다.
    """
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            sleep_random()

            headers = HEADERS.copy()
            if referer:
                headers["Referer"] = referer

            resp = session.get(url, headers=headers, timeout=timeout)

            if resp.status_code in [400, 403, 429]:
                last_error = requests.HTTPError(
                    f"{resp.status_code} Client Error: {resp.reason} for url: {url}"
                )

                print(f"[요청 거절 {attempt}/{MAX_RETRIES}] {url} / status={resp.status_code}")

                # 목록페이지를 먼저 열어 Referer / 세션 흐름을 맞춘 뒤 재시도
                if referer and WARMUP_REFERER_BEFORE_DETAIL:
                    try:
                        time.sleep(2)
                        session.get(referer, headers=HEADERS, timeout=timeout)
                    except Exception:
                        pass

                time.sleep(BLOCK_STATUS_SLEEP * attempt)
                continue

            resp.raise_for_status()

            if resp.apparent_encoding:
                resp.encoding = resp.apparent_encoding
            elif not resp.encoding or resp.encoding.lower() == "iso-8859-1":
                resp.encoding = "euc-kr"

            return BeautifulSoup(resp.text, "html.parser")

        except Exception as e:
            last_error = e
            print(f"[요청 재시도 {attempt}/{MAX_RETRIES}] {url} / {e}")
            time.sleep(RETRY_SLEEP * attempt)

    raise last_error

#상품코드 및 납품사례ID, 수집키 가져오기
def get_hidden_value(soup_or_tag, input_id: str):
    tag = soup_or_tag.find("input", {"id": input_id})
    return tag.get("value", "").strip() if tag else ""


def clean_label_value(text: str, label: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(rf"^{re.escape(label)}\s*[:：]\s*", "", text).strip()
    return text


def clean_price_to_int(text: str):
    if text is None:
        return None

    text = str(text).strip()

    if not text:
        return None

    if "가격문의" in text:
        return "가격문의"

    num = re.sub(r"[^0-9]", "", text)
    return int(num) if num else None


def parse_html_category_text(cat_html: str) -> str:
    if not cat_html:
        return ""

    decoded = html.unescape(cat_html)
    cat_soup = BeautifulSoup(decoded, "html.parser")
    parts = [a.get_text(" ", strip=True) for a in cat_soup.find_all("a")]
    parts = [p for p in parts if p]

    if parts:
        return " > ".join(parts)

    return cat_soup.get_text(" ", strip=True).replace(">", " > ")


def split_buyer_category(text: str):
    """
    판촉사랑 목록의 업종/행사 값을 구매처분류(중/소/세)로 나눕니다.

    예: 학교/교육기관 > 대학교/대학원 > 행사/홍보
    -> 구매처분류(중)=학교/교육기관, 구매처분류(소)=대학교/대학원, 구매처분류(세)=행사/홍보
    """
    text = str(text or "").strip()

    if not text:
        return "", "", ""

    parts = [p.strip() for p in text.split(">")]
    parts = [p for p in parts if p]
    parts = parts + [""] * 3

    return parts[0], parts[1], parts[2]


def add_buyer_category_to_row(row: dict) -> dict:
    """
    row의 업종/행사 값을 기준으로 구매처분류(중/소/세)를 추가합니다.
    """
    buyer_mid, buyer_small, buyer_detail = split_buyer_category(row.get("업종/행사", ""))
    row["구매처분류(중)"] = buyer_mid
    row["구매처분류(소)"] = buyer_small
    row["구매처분류(세)"] = buyer_detail
    return row


def ensure_buyer_category_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    기존 LIST_ONLY_CSV 또는 CHECKPOINT_CSV에 구매처분류 컬럼이 없더라도
    업종/행사 컬럼을 기준으로 다시 생성해서 이어하기가 가능하게 합니다.
    """
    df = df.copy()

    for col in BUYER_CATEGORY_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    if "업종/행사" in df.columns:
        buyer_values = df["업종/행사"].fillna("").apply(split_buyer_category)
        df["구매처분류(중)"] = buyer_values.apply(lambda x: x[0])
        df["구매처분류(소)"] = buyer_values.apply(lambda x: x[1])
        df["구매처분류(세)"] = buyer_values.apply(lambda x: x[2])

    return df


def make_collect_key(row: dict) -> str:
    case_id = str(row.get("납품사례ID", "") or "").strip()
    if case_id:
        return f"case:{case_id}"

    return "|".join([
        str(row.get("상품코드", "") or "").strip(),
        str(row.get("상품명", "") or "").strip(),
        str(row.get("등록일", "") or "").strip(),
        str(row.get("상세URL", "") or "").strip(),
    ])


def safe_write_csv(df: pd.DataFrame, path: str):
    """
    CSV를 현재 폴더에 저장한 뒤, 같은 파일을 상위폴더/dir에도 복사 저장합니다.
    """
    path = Path(path)
    tmp_path = Path(f"{path}.tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
    os.replace(tmp_path, path)

    # 현재 폴더 저장 후 dir 폴더에도 동일 파일 복사
    sync_to_dir(path)


def load_checkpoint() -> pd.DataFrame:
    if Path(CHECKPOINT_CSV).exists():
        df = pd.read_csv(CHECKPOINT_CSV, dtype=str)
        df = ensure_buyer_category_columns(df)
        for col in ALL_SAVE_COLUMNS:
            if col not in df.columns:
                df[col] = ""
        print(f"[이어하기] 기존 체크포인트 로드: {len(df)}건")
        return df[ALL_SAVE_COLUMNS]
    else:
        print("[새 작업] 기존 체크포인트 없음")
        return pd.DataFrame(columns=ALL_SAVE_COLUMNS)


def append_error_log(page, item_code, product_name, detail_url, error_message):
    error_row = pd.DataFrame([{
        "수집페이지": page,
        "상품코드": item_code,
        "상품명": product_name,
        "상세URL": detail_url,
        "오류내용": error_message,
        "오류시간": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    }])

    if Path(ERROR_LOG_CSV).exists():
        old = pd.read_csv(ERROR_LOG_CSV, dtype=str)
        error_df = pd.concat([old, error_row], ignore_index=True)
    else:
        error_df = error_row

    safe_write_csv(error_df, ERROR_LOG_CSV)


# =========================
# Selenium 상세페이지 수집 함수
# =========================

driver = None
selenium_detail_count = 0


def init_selenium_driver():
    """
    일반 Chrome Selenium 드라이버를 시작합니다.
    """
    global driver

    if driver is not None:
        return driver

    options = Options()

    if FAST_SELENIUM_MODE:
        # 전체 리소스 로딩 완료를 기다리지 않고 DOM이 준비되면 진행
        options.page_load_strategy = "eager"

        # 이미지 로딩 차단: 가격/분류/수량은 HTML 텍스트 기반이라 이미지가 필요 없음
        prefs = {
            "profile.managed_default_content_settings.images": 2,
            "profile.default_content_setting_values.notifications": 2,
        }
        options.add_experimental_option("prefs", prefs)


    if SELENIUM_HEADLESS:
        options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-popup-blocking")
    options.add_argument("--lang=ko-KR")
    options.add_argument("--window-size=1400,1000")

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    driver.set_page_load_timeout(SELENIUM_PAGELOAD_TIMEOUT)

    return driver


def restart_selenium_driver():
    """
    장시간 실행 안정성을 위해 Selenium 브라우저를 재시작합니다.
    """
    global driver

    try:
        if driver is not None:
            driver.quit()
    except Exception:
        pass

    driver = None
    return init_selenium_driver()


def close_selenium_driver():
    """
    크롤링 종료 후 브라우저를 닫습니다.
    """
    global driver

    try:
        if driver is not None:
            driver.quit()
    except Exception:
        pass

    driver = None


def wait_detail_html_ready(drv, max_wait: float = None) -> str:
    """
    상세페이지 HTML에서 필요한 핵심 문구가 보이면 바로 반환합니다.
    기존처럼 무조건 몇 초씩 기다리지 않기 위한 빠른 대기 함수입니다.
    """
    max_wait = max_wait or SELENIUM_WAIT_SECONDS
    end_time = time.time() + max_wait

    last_html = ""

    while time.time() < end_time:
        try:
            html_source = drv.page_source or ""
            last_html = html_source

            if any(marker in html_source for marker in DETAIL_READY_MARKERS):
                return html_source

        except Exception:
            pass

        time.sleep(0.2)

    return last_html or (drv.page_source or "")


def fetch_soup_selenium(url: str, referer: str = None) -> BeautifulSoup:
    """
    Selenium으로 상세페이지를 열고 BeautifulSoup 객체로 반환합니다.

    2단계 상세 수집 전용입니다.
    목록페이지를 먼저 열고 상세페이지로 이동하도록 처리해서
    requests에서 발생하던 400 Bad Request를 줄입니다.
    """
    global selenium_detail_count

    drv = init_selenium_driver()
    selenium_detail_count += 1

    if SELENIUM_RESTART_EVERY and selenium_detail_count % SELENIUM_RESTART_EVERY == 0:
        print(f"[Selenium 재시작] 상세페이지 {selenium_detail_count}건 처리")
        drv = restart_selenium_driver()

    time.sleep(random.uniform(SELENIUM_SLEEP_MIN, SELENIUM_SLEEP_MAX))

    try:
        if referer:
            drv.get(referer)
            time.sleep(random.uniform(0.8, 1.5))

        drv.get(url)
        html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        if "Bad Request" in html_source or "400" in drv.title:
            print(f"[Selenium 400 감지] 재시도: {url}")
            if referer:
                drv.get(referer)
                time.sleep(random.uniform(1.0, 2.0))
            drv.get(url)
            html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        return BeautifulSoup(html_source, "html.parser")

    except TimeoutException:
        print(f"[Selenium Timeout] {url}")
        try:
            drv.execute_script("window.stop();")
        except Exception:
            pass

        html_source = drv.page_source or ""
        if html_source:
            return BeautifulSoup(html_source, "html.parser")

        raise

    except WebDriverException as e:
        print(f"[Selenium 오류] 브라우저 재시작 후 1회 재시도: {e}")
        drv = restart_selenium_driver()

        if referer:
            drv.get(referer)
            time.sleep(random.uniform(1.0, 2.0))

        drv.get(url)
        html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        return BeautifulSoup(html_source or "", "html.parser")


def fetch_soup_selenium_direct(url: str) -> BeautifulSoup:
    """
    LIST_ONLY_CSV의 상세URL(I열)을 Selenium으로 직접 열고 BeautifulSoup으로 반환합니다.

    납품사례 목록페이지를 먼저 열지 않습니다.
    """
    global selenium_detail_count

    drv = init_selenium_driver()
    selenium_detail_count += 1

    if SELENIUM_RESTART_EVERY and selenium_detail_count % SELENIUM_RESTART_EVERY == 0:
        print(f"[Selenium 재시작] 상세페이지 {selenium_detail_count}건 처리")
        drv = restart_selenium_driver()

    time.sleep(random.uniform(SELENIUM_SLEEP_MIN, SELENIUM_SLEEP_MAX))

    try:
        drv.get(url)
        html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        # Selenium에서도 400이 뜨면 1회만 다시 직접 접속
        if "Bad Request" in html_source or "400" in drv.title:
            print(f"[Selenium 400 감지] 직접 URL 재시도: {url}")
            time.sleep(random.uniform(3.0, 6.0))
            drv.get(url)
            html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        return BeautifulSoup(html_source, "html.parser")

    except TimeoutException:
        print(f"[Selenium Timeout] {url}")
        try:
            drv.execute_script("window.stop();")
        except Exception:
            pass

        html_source = drv.page_source or ""
        if html_source:
            return BeautifulSoup(html_source, "html.parser")

        raise

    except WebDriverException as e:
        print(f"[Selenium 오류] 브라우저 재시작 후 직접 URL 1회 재시도: {e}")
        drv = restart_selenium_driver()
        drv.get(url)
        html_source = wait_detail_html_ready(drv, SELENIUM_WAIT_SECONDS)

        return BeautifulSoup(html_source or "", "html.parser")


## 3단계. 목록 페이지에서 기본 정보 뽑는 함수 준비

이 셀은 납품사례 목록 페이지에서 보이는 기본 정보를 추출하는 함수입니다.

수집하는 주요 값은 아래입니다.

- 상품코드
- 상품명
- 인쇄방법
- 업종/행사
- 등록일
- 상세URL
- 납품사례ID
- 수집키

중요한 점은 이 단계에서는 상세페이지에 들어가지 않습니다.
목록 화면에 보이는 값과 상세페이지 링크만 먼저 안전하게 모읍니다.


In [ ]:
# =========================
# 목록 페이지 파싱
# =========================

def extract_direct_text(tag):
    """태그의 직계 텍스트만 추출합니다. textarea/하위 테이블에 들어있는 숨은 HTML이 섞이는 문제를 방지합니다."""
    if tag is None:
        return ""
    parts = []
    for s in tag.find_all(string=True, recursive=False):
        txt = str(s).strip()
        if txt:
            parts.append(txt)
    return re.sub(r"\s+", " ", " ".join(parts)).strip()


def find_product_detail_link(card, item_code: str):
    """
    상품명 영역의 a href를 찾아 상세페이지 URL을 반환합니다.

    기준 HTML 예시:
    <a href="/shop/viewitem.asp?mcd=M00000001&ca=168&itemcd=04130299" target="_blank">
        <span>상품명</span>
    </a>
    """
    if not card:
        return "", ""

    # 1순위: itemcd가 href 안에 포함된 상품 상세 링크
    for a in card.find_all("a", href=True):
        href = html.unescape(a.get("href", "")).strip()
        text = a.get_text(" ", strip=True)

        if "viewitem.asp" in href and f"itemcd={item_code}" in href:
            return urljoin(BASE_URL, href), text

    # 2순위: viewitem.asp가 포함된 첫 번째 상세 링크
    for a in card.find_all("a", href=True):
        href = html.unescape(a.get("href", "")).strip()
        text = a.get_text(" ", strip=True)

        if "viewitem.asp" in href:
            return urljoin(BASE_URL, href), text

    return "", ""


def extract_print_method_from_card(card, item_code: str):
    """
    목록 카드에서 인쇄방법만 정확히 추출합니다.

    기준 구조:
    <tr height="30">
        <td width="40%">상품코드 : 04130299</td>
        <td width="60%">인쇄방법 : 실크인쇄</td>
    </tr>

    기존처럼 card 전체 text를 읽으면 숨겨진 textarea의 HTML까지 섞일 수 있으므로
    반드시 '상품코드 td'의 다음 형제 td에서만 추출합니다.
    """
    if not card:
        return ""

    # 1순위: 상품코드가 들어있는 td를 찾고, 바로 옆 td에서 인쇄방법 추출
    code_patterns = [
        f"상품코드 : {item_code}",
        f"상품코드: {item_code}",
    ]

    for td in card.find_all("td"):
        direct_text = extract_direct_text(td)
        if any(p in direct_text for p in code_patterns):
            next_td = td.find_next_sibling("td")
            if next_td:
                method_text = extract_direct_text(next_td)
                if not method_text:
                    method_text = next_td.get_text(" ", strip=True)

                method = clean_label_value(method_text, "인쇄방법")
                method = re.sub(r"\s+", " ", method).strip()

                # 인쇄방법이 비어있거나 라벨만 있으면 빈값
                if method and method != "인쇄방법":
                    return method
            break

    # 2순위: 직계 텍스트가 '인쇄방법 :'으로 시작하는 td만 스캔
    for td in card.find_all("td"):
        direct_text = extract_direct_text(td)
        m = re.search(r"^인쇄방법\s*[:：]\s*(.+)$", direct_text)
        if m:
            method = m.group(1).strip()
            return method

    return ""


def parse_list_page(list_url: str, page: int):
    soup = fetch_soup(list_url)

    rows = []

    item_inputs = soup.select("input[id^='itemcd_']")

    print(f"[목록] page={page} / itemcd input 수: {len(item_inputs)}")

    for inp in item_inputs:
        item_input_id = inp.get("id", "")
        idx = item_input_id.replace("itemcd_", "").strip()
        if not idx.isdigit():
            continue

        item_code = inp.get("value", "").strip()
        case_id = get_hidden_value(soup, f"idx_{idx}")
        product_name = get_hidden_value(soup, f"itemnm_{idx}")
        mcd = get_hidden_value(soup, f"mcd_{idx}")
        ca = get_hidden_value(soup, f"ca_{idx}")
        catnm = get_hidden_value(soup, f"catnm_{idx}")
        event_category = parse_html_category_text(catnm)

        # hidden input 바로 뒤의 상품 카드 td
        card = inp.find_next("td", attrs={"width": "33%"})

        registered_date = ""
        list_price_text = ""

        # 상세페이지는 반드시 상품명 a href에서 추출
        detail_url, product_name_from_link = find_product_detail_link(card, item_code)

        if product_name_from_link:
            product_name = product_name_from_link

        # 인쇄방법은 상품코드 td의 다음 td에서만 추출
        print_method = extract_print_method_from_card(card, item_code)

        if card:
            # 등록일
            date_tag = card.find(string=lambda s: s and "등록일" in s)
            if date_tag:
                registered_date = clean_label_value(str(date_tag), "등록일")
            else:
                card_text = card.get_text(" ", strip=True)
                m = re.search(r"등록일\s*[:：]\s*(\d{4}-\d{2}-\d{2})", card_text)
                registered_date = m.group(1) if m else ""

            # 목록 하단 가격
            strongs = [s.get_text(" ", strip=True) for s in card.find_all("strong")]
            if strongs:
                list_price_text = strongs[-1].replace("~", "").strip()

        # a href 추출 실패 시에만 hidden 값으로 상세 URL 보정
        if not detail_url and item_code and mcd and ca:
            detail_url = urljoin(BASE_URL, f"/shop/viewitem.asp?mcd={mcd}&ca={ca}&itemcd={item_code}")

        buyer_mid, buyer_small, buyer_detail = split_buyer_category(event_category)

        row = {
            "납품사례ID": case_id,
            "상품코드": item_code,
            "구매처분류(중)": buyer_mid,
            "구매처분류(소)": buyer_small,
            "구매처분류(세)": buyer_detail,
            "상품명": product_name,
            "인쇄방법": print_method,  # 비어있으면 그대로 빈칸 저장
            "업종/행사": event_category,
            "등록일": registered_date,
            "목록가격": list_price_text,
            "상세URL": detail_url,
            "수집페이지": page,
        }
        row["수집키"] = make_collect_key(row)

        rows.append(row)

    return rows

## 4단계. 상세페이지에서 상품분류, 최소수량, 가격 뽑는 함수 준비

이 셀은 상세페이지에 들어가서 목록 페이지에는 없는 정보를 추출하는 함수입니다.

수집하는 주요 값은 아래입니다.

- 상품분류(대)
- 상품분류(중)
- 상품분류(소)
- 최소인쇄수량
- 최소주문수량
- 대량가격(원)
- 중간가격(원)
- 소량가격(원)

가격표 구조가 상품마다 조금씩 다를 수 있어서 여러 방식으로 가격을 찾도록 되어 있습니다.
가격표가 없고 실제로 `가격문의`인 경우에는 `가격문의`로 처리합니다.


In [ ]:
# =========================
# 상세 페이지 파싱
# =========================

def parse_product_categories(detail_soup: BeautifulSoup):
    """
    상세페이지 현재위치 영역에서 상품분류(대/중/소)를 추출합니다.

    기준:
    현재위치 : HOME > 부채 > 부채 > 부채(전통,한지부채)
    """

    position_td = None

    # 1순위: 현재위치 아이콘 arr01.gif가 들어있는 td
    img = detail_soup.find("img", src=lambda s: s and "/image/sub/arr01.gif" in s)
    if img:
        position_td = img.find_parent("td")

    # 2순위: 텍스트에 '현재위치'가 있는 td
    if position_td is None:
        for td in detail_soup.find_all("td"):
            td_text = td.get_text(" ", strip=True)
            if "현재위치" in td_text:
                position_td = td
                break

    categories = []

    if position_td is not None:
        for a in position_td.find_all("a", href=True):
            href = html.unescape(a.get("href", "")).strip()
            name = a.get_text(" ", strip=True)

            if not name or name.upper() == "HOME":
                continue

            if (
                "/shop/main.asp" in href
                or "/shop/middle.asp" in href
                or "/shop/small.asp" in href
            ):
                categories.append(name)

    # 3순위: 현재위치 텍스트만 있는 경우 보조 추출
    if not categories:
        page_text = detail_soup.get_text(" ", strip=True)
        m = re.search(r"현재위치\s*:\s*HOME\s*>\s*([^>\n]+)\s*>\s*([^>\n]+)\s*>\s*([^>\n]+)", page_text)
        if m:
            categories = [m.group(1).strip(), m.group(2).strip(), m.group(3).strip()]

    large = categories[0] if len(categories) >= 1 else ""
    middle = categories[1] if len(categories) >= 2 else ""
    small = categories[2] if len(categories) >= 3 else ""

    return large, middle, small


def parse_minimum_quantities(detail_soup: BeautifulSoup):
    """
    상세페이지에서 최소인쇄수량 / 최소주문수량을 각각 독립적으로 추출합니다.

    예:
    - 최소인쇄수량 : 100개
    - 최소주문수량 : 100개
    - 최소인쇄수량 : 20세트
    - 최소주문수량 : 10세트
    """

    min_print_qty = ""
    min_order_qty = ""

    page_text = detail_soup.get_text("\n", strip=True)
    page_text = html.unescape(page_text)
    page_text = re.sub(r"\s+", " ", page_text)

    # 숫자 + 단위 추출
    # 단위는 다음 라벨 전까지 너무 많이 먹지 않도록 짧게 제한
    qty_pattern = r"([0-9,]+)\s*([가-힣A-Za-z0-9()\/·\-\+]+)?"

    m_print = re.search(
        rf"최소\s*인쇄\s*수량\s*[:：]\s*{qty_pattern}",
        page_text
    )
    if m_print:
        number = m_print.group(1)
        unit = m_print.group(2) or ""
        min_print_qty = f"{number}{unit}"

    m_order = re.search(
        rf"최소\s*주문\s*수량\s*[:：]\s*{qty_pattern}",
        page_text
    )
    if m_order:
        number = m_order.group(1)
        unit = m_order.group(2) or ""
        min_order_qty = f"{number}{unit}"

    if not min_order_qty:
        m_order_alt = re.search(
            rf"최소\s*주문\s*[:：]\s*{qty_pattern}",
            page_text
        )
        if m_order_alt:
            number = m_order_alt.group(1)
            unit = m_order_alt.group(2) or ""
            min_order_qty = f"{number}{unit}"

    return min_print_qty, min_order_qty


def decode_percent_u_escaped(text: str) -> str:
    """
    가격표가 document.write(unescape("...")) 안에 들어있는 경우를 처리합니다.
    Python의 urllib.parse.unquote는 %uC218 같은 JS escape를 처리하지 못하므로 별도 처리합니다.
    """
    from urllib.parse import unquote

    if not text:
        return ""

    def repl_unicode(m):
        return chr(int(m.group(1), 16))

    text = re.sub(r"%u([0-9A-Fa-f]{4})", repl_unicode, text)
    text = unquote(text)

    return text


def find_price_tables_in_soup(soup: BeautifulSoup):
    """
    상세페이지에서 가격표 table 후보를 찾습니다.

    기준:
    - table 텍스트에 '즉시 할인가' 또는 '일반 판매가'가 있는 경우
    - 또는 table 내부에 input name='rateprice'가 있는 경우
    """
    tables = []

    for table in soup.find_all("table"):
        table_text = table.get_text(" ", strip=True)
        has_price_label = ("즉시 할인가" in table_text) or ("즉시할인가" in table_text) or ("일반 판매가" in table_text) or ("일반판매가" in table_text)
        has_rateprice = table.find("input", {"name": "rateprice"}) is not None

        if has_price_label or has_rateprice:
            tables.append(table)

    return tables


def extract_price_values_from_table(table):
    """
    가격표 table에서 가격을 추출합니다.

    우선순위:
    1. '즉시 할인가' 행
    2. '일반 판매가' 행
    3. table 내부 hidden input name='rateprice'

    기준:
    - 1번째 가격 = 소량가격
    - 4번째 가격 = 중간가격
    - 마지막 가격 = 대량가격

    가격 컬럼이 7개든 8개든 마지막 가격을 대량가격으로 사용합니다.
    """
    table_text = table.get_text(" ", strip=True)

    price_rows = {}

    for tr in table.find_all("tr"):
        cells = [td.get_text(" ", strip=True) for td in tr.find_all("td")]
        if not cells:
            continue

        row_name = re.sub(r"\s+", "", cells[0])

        if "즉시할인가" in row_name:
            price_rows["즉시할인가"] = cells[1:]
        elif "일반판매가" in row_name:
            price_rows["일반판매가"] = cells[1:]

    raw_prices = price_rows.get("즉시할인가") or price_rows.get("일반판매가") or []

    # 가격 행에 가격문의가 명시된 경우만 가격문의로 처리
    if raw_prices and any("가격문의" in str(x) for x in raw_prices):
        return "가격문의", "가격문의", "가격문의"

    prices = [clean_price_to_int(x) for x in raw_prices]
    prices = [p for p in prices if isinstance(p, int)]

    # table 내부 hidden rateprice 보조 추출
    if not prices:
        for inp in table.find_all("input", {"name": "rateprice"}):
            value = inp.get("value", "")
            price = clean_price_to_int(value)
            if isinstance(price, int):
                prices.append(price)

    if prices:
        small_price = prices[0]
        middle_price = prices[3] if len(prices) >= 4 else prices[len(prices) // 2]
        large_price = prices[-1]
        return large_price, middle_price, small_price

    # 진짜 가격문의 테이블인 경우만 가격문의 처리
    if "가격문의" in table_text and not re.search(r"[0-9][0-9,]*\s*원", table_text):
        return "가격문의", "가격문의", "가격문의"

    return None, None, None


def parse_price_columns(detail_soup: BeautifulSoup):
    """
    상세페이지 가격표에서 가격을 추출합니다.

    기준:
    - 즉시 할인가 행 우선
    - 없으면 일반 판매가 행 사용
    - 1번째 가격 = 소량가격
    - 4번째 가격 = 중간가격
    - 마지막 가격 = 대량가격

    주의:
    페이지 어딘가에 '가격문의' 문구가 있어도,
    가격표나 rateprice 값이 있으면 실제 가격을 우선 사용합니다.
    """
    page_text = detail_soup.get_text(" ", strip=True)
    html_text = str(detail_soup)

    # 1순위: HTML에 이미 렌더링된 table이 있는 경우
    for table in find_price_tables_in_soup(detail_soup):
        large_price, middle_price, small_price = extract_price_values_from_table(table)
        if any(v is not None for v in [large_price, middle_price, small_price]):
            return large_price, middle_price, small_price

    # 2순위: document.write(unescape("...")) script 내부에 가격표가 있는 경우
    patterns = [
        r'document\.write\s*\(\s*unescape\s*\(\s*"([^"]+)"\s*\)\s*\)',
        r"document\.write\s*\(\s*unescape\s*\(\s*'([^']+)'\s*\)\s*\)",
    ]

    for pattern in patterns:
        for m in re.finditer(pattern, html_text, flags=re.IGNORECASE | re.DOTALL):
            encoded_html = m.group(1)
            decoded_html = decode_percent_u_escaped(encoded_html)

            if not decoded_html:
                continue

            price_soup = BeautifulSoup(decoded_html, "html.parser")

            for table in find_price_tables_in_soup(price_soup):
                large_price, middle_price, small_price = extract_price_values_from_table(table)
                if any(v is not None for v in [large_price, middle_price, small_price]):
                    return large_price, middle_price, small_price

            # decoded 내부에 진짜 가격문의만 있는 경우
            decoded_text = price_soup.get_text(" ", strip=True)
            if "가격문의" in decoded_text and not re.search(r"[0-9][0-9,]*\s*원", decoded_text):
                return "가격문의", "가격문의", "가격문의"

    # 3순위: 상세페이지에 rateprice hidden만 있는 경우
    rate_prices = []
    for inp in detail_soup.find_all("input", {"name": "rateprice"}):
        value = inp.get("value", "")
        price = clean_price_to_int(value)

        if isinstance(price, int):
            rate_prices.append(price)

    # 3-1순위: raw HTML 문자열에서 rateprice 직접 추출
    if not rate_prices:
        for m in re.finditer(r'name=["\']rateprice["\']\s+value=["\']([^"\']+)["\']', html_text, flags=re.IGNORECASE):
            price = clean_price_to_int(m.group(1))
            if isinstance(price, int):
                rate_prices.append(price)

    if rate_prices:
        small_price = rate_prices[0]
        middle_price = rate_prices[3] if len(rate_prices) >= 4 else rate_prices[len(rate_prices) // 2]
        large_price = rate_prices[-1]
        return large_price, middle_price, small_price

    # 4순위: 진짜 가격문의 페이지인 경우만 가격문의 처리
    # 단순히 페이지 어딘가에 가격문의 문구가 있는 것만으로는 가격문의 처리하지 않음
    has_price_number = bool(re.search(r"[0-9][0-9,]*\s*원", page_text))
    if "가격문의" in page_text and not has_price_number:
        return "가격문의", "가격문의", "가격문의"

    return None, None, None


def parse_detail_page(detail_url: str, referer: str = None):
    if not detail_url:
        return {
            "상품분류(대)": "",
            "상품분류(중)": "",
            "상품분류(소)": "",
            "최소인쇄수량": "",
            "최소주문수량": "",
            "대량가격(원)": None,
            "중간가격(원)": None,
            "소량가격(원)": None,
        }

    if DETAIL_FETCH_MODE == "selenium":
        if DETAIL_DIRECT_URL_ONLY:
            soup = fetch_soup_selenium_direct(detail_url)
        else:
            soup = fetch_soup_selenium(detail_url, referer=referer)
    else:
        soup = fetch_soup(detail_url, referer=referer)

    large_cat, middle_cat, small_cat = parse_product_categories(soup)
    min_print_qty, min_order_qty = parse_minimum_quantities(soup)
    large_price, middle_price, small_price = parse_price_columns(soup)

    return {
        "상품분류(대)": large_cat,
        "상품분류(중)": middle_cat,
        "상품분류(소)": small_cat,
        "최소인쇄수량": min_print_qty,
        "최소주문수량": min_order_qty,
        "대량가격(원)": large_price,
        "중간가격(원)": middle_price,
        "소량가격(원)": small_price,
    }

## 5단계. 체크포인트 데이터를 엑셀로 저장하는 함수 준비

이 셀은 중간 저장 파일인 `87sarang_supplycase_checkpoint.csv`를 읽어서 최종 엑셀을 만드는 함수입니다.

주요 역할은 아래입니다.

- 템플릿 엑셀 파일이 있으면 그 양식을 사용합니다.
- 템플릿이 없으면 기본 엑셀 양식을 새로 만듭니다.
- 체크포인트 데이터를 최종 엑셀 컬럼 순서에 맞춰 저장합니다.
- 헤더 색상, 글꼴, 테두리, 필터, 고정틀 등을 적용합니다.

실제 저장은 뒤쪽의 `save_excel_from_checkpoint()` 실행 셀에서 합니다.


In [ ]:
# =========================
# 엑셀 저장 함수
# =========================

def create_fallback_workbook():
    wb = Workbook()
    ws = wb.active
    ws.title = "납품사례_분류가격"

    ws.append(COLUMNS)

    header_fill = PatternFill("solid", fgColor="1F4E79")
    header_font = Font(name="맑은 고딕", size=10, bold=True, color="FFFFFF")
    thin_gray = Side(style="thin", color="D9E2F3")

    for col_idx, col_name in enumerate(COLUMNS, 1):
        cell = ws.cell(1, col_idx)
        cell.value = col_name
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = Border(
            left=thin_gray,
            right=thin_gray,
            top=thin_gray,
            bottom=thin_gray
        )

    widths = [5, 12, 18, 18, 18, 45, 20, 22, 22, 13, 13, 13, 13, 13, 15, 35, 13, 15, 45]
    for i, width in enumerate(widths, 1):
        ws.column_dimensions[ws.cell(1, i).column_letter].width = width

    ws.row_dimensions[1].height = 22
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{ws.cell(1, len(COLUMNS)).column_letter}1"

    return wb, ws


def save_excel_from_checkpoint(output_xlsx=OUTPUT_XLSX):
    checkpoint_df = load_checkpoint()

    if checkpoint_df.empty:
        print("[엑셀 저장 생략] 체크포인트 데이터가 없습니다.")
        return

    export_df = checkpoint_df.copy()

    for col in COLUMNS:
        if col not in export_df.columns:
            export_df[col] = ""

    export_df = export_df.reset_index(drop=True)
    export_df["No"] = range(1, len(export_df) + 1)
    export_df = export_df[COLUMNS]

    template_path = Path(TEMPLATE_XLSX)

    thin_gray = Side(style="thin", color="D9D9D9")
    body_border = Border(
        left=thin_gray,
        right=thin_gray,
        top=thin_gray,
        bottom=thin_gray
    )
    body_font = Font(name="맑은 고딕", size=10, color="000000")
    body_alignment = Alignment(vertical="center", wrap_text=False)
    no_fill = PatternFill(fill_type=None)

    if template_path.exists():
        wb = load_workbook(template_path)
        ws = wb.active

        # 기존 템플릿의 1행 헤더 스타일을 기준으로 보존
        # 새로 추가된 구매처분류 컬럼 등은 A1 스타일을 기준으로 적용
        header_styles = []
        header_fonts = []
        header_fills = []
        header_borders = []
        header_alignments = []

        for c in range(1, len(COLUMNS) + 1):
            source_col = c if c <= ws.max_column else 1
            header_styles.append(copy(ws.cell(1, source_col)._style))
            header_fonts.append(copy(ws.cell(1, source_col).font))
            header_fills.append(copy(ws.cell(1, source_col).fill))
            header_borders.append(copy(ws.cell(1, source_col).border))
            header_alignments.append(copy(ws.cell(1, source_col).alignment))

        if ws.max_row > 1:
            ws.delete_rows(2, ws.max_row - 1)

        for c, col_name in enumerate(COLUMNS, 1):
            cell = ws.cell(1, c)
            cell.value = col_name
            cell._style = copy(header_styles[c - 1])
            cell.font = copy(header_fonts[c - 1])
            cell.fill = copy(header_fills[c - 1])
            cell.border = copy(header_borders[c - 1])
            cell.alignment = copy(header_alignments[c - 1])

    else:
        wb, ws = create_fallback_workbook()

    for r_idx, record in enumerate(export_df.to_dict("records"), start=2):
        for c_idx, col_name in enumerate(COLUMNS, start=1):
            cell = ws.cell(r_idx, c_idx)
            value = record.get(col_name)

            if pd.isna(value):
                value = None

            # 가격 컬럼 처리
            # 가격문의는 문자열 그대로 유지하고, 숫자는 숫자로 저장
            if col_name in ["대량가격(원)", "중간가격(원)", "소량가격(원)"]:
                if value not in [None, ""]:
                    value_str = str(value).strip()

                    if "가격문의" in value_str:
                        value = "가격문의"
                    else:
                        value_str = value_str.replace(",", "").replace("원", "").strip()
                        value = int(value_str) if value_str.isdigit() else None

            cell.value = value

            # 2행부터는 배경색 제거
            cell.fill = no_fill
            cell.font = body_font
            cell.border = body_border
            cell.alignment = body_alignment

    price_cols = [COLUMNS.index(col) + 1 for col in ["대량가격(원)", "중간가격(원)", "소량가격(원)"]]
    text_cols = [
        COLUMNS.index(col) + 1
        for col in [
            "상품코드",
            "구매처분류(중)", "구매처분류(소)", "구매처분류(세)",
            "최소인쇄수량", "최소주문수량",
            "인쇄방법", "업종/행사", "등록일",
            "납품사례ID", "수집키"
        ]
        if col in COLUMNS
    ]

    # 숫자/텍스트 형식
    for row in range(2, ws.max_row + 1):
        for col in price_cols:
            if ws.cell(row, col).value == "가격문의":
                ws.cell(row, col).number_format = '@'
            else:
                ws.cell(row, col).number_format = '#,##0'

        for col in text_cols:
            ws.cell(row, col).number_format = '@'

    default_widths = [5, 12, 18, 18, 18, 45, 20, 22, 22, 13, 13, 13, 13, 13, 15, 35, 13, 15, 45]
    for i, width in enumerate(default_widths, 1):
        col_letter = ws.cell(1, i).column_letter
        current_width = ws.column_dimensions[col_letter].width

        if current_width is None or current_width < 5:
            ws.column_dimensions[col_letter].width = width

    ws.freeze_panes = "A2"
    last_col_letter = ws.cell(1, len(COLUMNS)).column_letter
    ws.auto_filter.ref = f"A1:{last_col_letter}{max(ws.max_row, 1)}"

    wb.save(output_xlsx)

    # 현재 폴더 저장 후 dir 폴더에도 동일 파일 복사
    sync_to_dir(output_xlsx)

    print(f"[엑셀 저장 완료] {output_xlsx} / {len(export_df)}건")
    print(f"[dir 복사 완료] {SAVE_DIR / Path(output_xlsx).name}")


## 6단계. 1단계 목록 수집과 2단계 상세 수집 함수 준비

이 셀은 실제 수집 작업의 핵심 함수입니다.

포함된 주요 함수는 아래입니다.

- `load_list_only()`: 기존 목록 수집 파일이 있으면 불러옵니다.
- `save_list_only()`: 목록 수집 결과를 CSV로 저장합니다.
- `collect_list_only()`: 1단계 목록 수집을 실행합니다.
- `make_error_detail_row()`: 상세페이지 오류가 나도 기본 행을 남깁니다.
- `run_detail_from_list()`: 2단계 상세 수집을 실행합니다.

이 셀도 준비용 셀입니다. 이 셀을 실행해야 아래의 1단계 실행 셀과 2단계 실행 셀이 작동합니다.


In [ ]:
# =========================
# 1단계 목록 수집 / 2단계 상세 수집 함수
# =========================

def load_list_only() -> pd.DataFrame:
    """
    1단계 목록 수집 파일을 로드합니다.
    """
    if Path(LIST_ONLY_CSV).exists():
        df = pd.read_csv(LIST_ONLY_CSV, dtype=str).fillna("")
        df = ensure_buyer_category_columns(df)
        for col in LIST_COLUMNS:
            if col not in df.columns:
                df[col] = ""
        print(f"[목록 이어하기] 기존 목록 파일 로드: {len(df)}건")
        return df[LIST_COLUMNS]
    else:
        print("[목록 새 작업] 기존 목록 파일 없음")
        return pd.DataFrame(columns=LIST_COLUMNS)


def save_list_only(df: pd.DataFrame):
    """
    목록 수집 결과를 안전하게 저장합니다.
    """
    df = ensure_buyer_category_columns(df)

    for col in LIST_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    df = df[LIST_COLUMNS].copy()

    if not df.empty:
        df = df.drop_duplicates(subset=["수집키"], keep="first")

    safe_write_csv(df, LIST_ONLY_CSV)
    

def save_checkpoint_files(df: pd.DataFrame):
    """
    상세페이지 체크포인트를 CSV와 XLSX로 함께 저장합니다.
    CSV는 이어하기 기준 파일이고,
    XLSX는 확인용 파일입니다.
    """
    df = ensure_buyer_category_columns(df)

    for col in ALL_SAVE_COLUMNS:
        if col not in df.columns:
            df[col] = ""

    df = df[ALL_SAVE_COLUMNS].copy()

    # CSV 저장
    safe_write_csv(df, CHECKPOINT_CSV)

    # XLSX 확인용 저장
    df.to_excel(CHECKPOINT_XLSX, index=False, engine="openpyxl")

    # XLSX도 상위폴더/dir에 복사
    sync_to_dir(CHECKPOINT_XLSX)


def collect_list_only():
    """
    1단계:
    목록페이지에서 상품코드, 상품명, 인쇄방법, 업종/행사, 등록일, 상세URL만 먼저 수집합니다.
    상세페이지에는 들어가지 않습니다.
    """

    list_df = load_list_only()
    done_list_keys = set(list_df["수집키"].dropna().astype(str).tolist()) if not list_df.empty else set()

    new_rows = []

    try:
        for page in range(START_PAGE, END_PAGE + 1):
            page_url = build_page_url(START_URL, page)

            print(f"\n[1단계 목록 수집] page={page}")
            print(f"  URL: {page_url}")

            try:
                page_rows = parse_list_page(page_url, page)

            except Exception as e:
                err_msg = f"{type(e).__name__}: {e}"
                print(f"[목록 오류] page={page} / {err_msg}")
                append_error_log(page, "", "", page_url, f"목록 페이지 오류: {err_msg}")
                continue

            if not page_rows:
                print(f"[중단] page={page}에서 수집 항목이 없습니다.")
                break

            page_new_count = 0

            for row in page_rows:
                collect_key = str(row.get("수집키", "") or "").strip()

                if not collect_key:
                    collect_key = make_collect_key(row)

                if collect_key in done_list_keys:
                    print(f"[목록 건너뜀] 이미 목록 수집됨: {row.get('상품코드')} / {str(row.get('상품명', ''))[:35]}")
                    continue

                buyer_mid, buyer_small, buyer_detail = split_buyer_category(row.get("업종/행사", ""))

                list_row = {
                    "수집키": collect_key,
                    "납품사례ID": row.get("납품사례ID", ""),
                    "수집페이지": page,
                    "상품코드": row.get("상품코드", ""),
                    "구매처분류(중)": buyer_mid,
                    "구매처분류(소)": buyer_small,
                    "구매처분류(세)": buyer_detail,
                    "상품명": row.get("상품명", ""),
                    "인쇄방법": row.get("인쇄방법", ""),
                    "업종/행사": row.get("업종/행사", ""),
                    "등록일": row.get("등록일", ""),
                    "상세URL": row.get("상세URL", ""),
                    "목록수집상태": "완료",
                }

                new_rows.append(list_row)
                done_list_keys.add(collect_key)
                page_new_count += 1

            if new_rows:
                list_df = pd.concat([list_df, pd.DataFrame(new_rows)], ignore_index=True)
                list_df = list_df.drop_duplicates(subset=["수집키"], keep="first")
                save_list_only(list_df)
                new_rows = []

            print(f"  -> page={page} 목록 저장 완료 / 이번 페이지 신규 {page_new_count}건 / 누적 {len(list_df)}건")

    except KeyboardInterrupt:
        print("\n[목록 수집 중지] 현재까지 목록 데이터를 저장합니다.")
        if new_rows:
            list_df = pd.concat([list_df, pd.DataFrame(new_rows)], ignore_index=True)
        save_list_only(list_df)
        print("[목록 중지 저장 완료]")
        return list_df

    if new_rows:
        list_df = pd.concat([list_df, pd.DataFrame(new_rows)], ignore_index=True)

    save_list_only(list_df)

    print(f"\n[1단계 목록 수집 완료] 총 {len(list_df)}건 저장")
    print(f"저장 파일: {LIST_ONLY_CSV}")

    return list_df


def make_error_detail_row(row, collect_key, item_code, product_name, detail_url, page):
    """
    상세페이지 오류가 나도 목록에서 확보한 기본정보를 최종 체크포인트에 남기기 위한 행입니다.
    """
    buyer_mid, buyer_small, buyer_detail = split_buyer_category(row.get("업종/행사", ""))

    return {
        "No": "",
        "상품코드": item_code,
        "구매처분류(중)": buyer_mid,
        "구매처분류(소)": buyer_small,
        "구매처분류(세)": buyer_detail,
        "상품명": product_name,
        "상품분류(대)": "",
        "상품분류(중)": "",
        "상품분류(소)": "",
        "최소인쇄수량": "",
        "최소주문수량": "",
        "대량가격(원)": "",
        "중간가격(원)": "",
        "소량가격(원)": "",
        "인쇄방법": row.get("인쇄방법", ""),
        "업종/행사": row.get("업종/행사", ""),
        "등록일": row.get("등록일", ""),
        "수집키": collect_key,
        "납품사례ID": row.get("납품사례ID", ""),
        "수집페이지": page,
        "상세URL": detail_url,
        "수집상태": "상세페이지 오류",
    }


def run_detail_from_list():
    """
    2단계:
    1단계에서 저장한 LIST_ONLY_CSV를 읽고,
    상세URL에 하나씩 접속해서 상품분류, 최소수량, 가격을 수집합니다.
    """

    if not Path(LIST_ONLY_CSV).exists():
        raise FileNotFoundError(
            f"{LIST_ONLY_CSV} 파일이 없습니다. 먼저 collect_list_only()를 실행하세요."
        )

    list_df = pd.read_csv(LIST_ONLY_CSV, dtype=str).fillna("")
    list_df = ensure_buyer_category_columns(list_df)

    for col in LIST_COLUMNS:
        if col not in list_df.columns:
            list_df[col] = ""

    list_df = list_df[LIST_COLUMNS].copy()

    checkpoint_df = load_checkpoint()

    # RETRY_DETAIL_ERRORS=True이면 기존 상세페이지 오류 행은 다시 시도할 수 있도록 제거
    if RETRY_DETAIL_ERRORS and not checkpoint_df.empty:
        error_keys = set(
            checkpoint_df.loc[
                checkpoint_df["수집상태"].astype(str) == "상세페이지 오류",
                "수집키"
            ].dropna().astype(str).tolist()
        )

        if error_keys:
            print(f"[오류 재시도] 기존 상세페이지 오류 {len(error_keys)}건을 재시도 대상으로 전환")
            checkpoint_df = checkpoint_df[~checkpoint_df["수집키"].astype(str).isin(error_keys)].copy()
            save_checkpoint_files(checkpoint_df)
    # RETRY_BLANK_DETAIL_ROWS=True이면 상세값이 거의 비어 있는 완료/오류 행도 다시 시도
    if RETRY_BLANK_DETAIL_ROWS and not checkpoint_df.empty:
        price_cols = ["대량가격(원)", "중간가격(원)", "소량가격(원)"]
        detail_cols = ["상품분류(대)", "상품분류(중)", "상품분류(소)", "최소인쇄수량", "최소주문수량"] + price_cols

        for col in detail_cols + ["수집상태"]:
            if col not in checkpoint_df.columns:
                checkpoint_df[col] = ""

        def is_blank_or_error_detail(row):
            status = str(row.get("수집상태", "") or "").strip()
            if status == "상세페이지 오류":
                return True

            values = [str(row.get(col, "") or "").strip() for col in detail_cols]
            # 가격문의는 실제 가격문의일 수 있으므로 단독으로는 재시도 조건으로 쓰지 않음
            # 다만 분류/최소수량/가격이 전부 빈칸이면 재시도
            return all(v == "" for v in values)

        blank_keys = set(
            checkpoint_df.loc[
                checkpoint_df.apply(is_blank_or_error_detail, axis=1),
                "수집키"
            ].dropna().astype(str).tolist()
        )

        if blank_keys:
            print(f"[빈 상세값 재시도] 기존 상세 오류/빈값 {len(blank_keys)}건을 재시도 대상으로 전환")
            checkpoint_df = checkpoint_df[~checkpoint_df["수집키"].astype(str).isin(blank_keys)].copy()
            save_checkpoint_files(checkpoint_df)

    done_keys = set()
    if not checkpoint_df.empty and "수집키" in checkpoint_df.columns:
        done_keys = set(checkpoint_df["수집키"].dropna().astype(str).tolist())

    total_new_count = 0
    processed_since_excel_save = 0

    try:
        for idx, row in list_df.iterrows():
            collect_key = str(row.get("수집키", "") or "").strip()
            item_code = str(row.get("상품코드", "") or "").strip()
            product_name = str(row.get("상품명", "") or "").strip()
            detail_url = str(row.get("상세URL", "") or "").strip()
            page = str(row.get("수집페이지", "") or "").strip()

            if not collect_key:
                collect_key = make_collect_key(row.to_dict())

            if not detail_url or "viewitem.asp" not in detail_url:
                err_msg = "상세URL 없음 또는 viewitem.asp URL 아님"
                print(f"[상세URL 오류] {item_code} / {product_name} / {err_msg}")
                append_error_log(page, item_code, product_name, detail_url, err_msg)
                continue

            if collect_key in done_keys:
                print(f"[상세 건너뜀] 이미 상세 수집됨: {item_code} / {product_name[:35]}")
                continue

            print(f"\n[2단계 상세 수집] page={page} / {item_code} / {product_name[:50]}")
            print(f"  상세URL: {detail_url}")

            try:
                # I열 상세URL을 Selenium으로 직접 접속해서 상세정보를 추출합니다.
                # 납품사례 목록주소를 먼저 열거나 링크를 타고 들어가지 않습니다.
                detail_data = parse_detail_page(detail_url, referer=None)

                buyer_mid, buyer_small, buyer_detail = split_buyer_category(row.get("업종/행사", ""))

                final_row = {
                    "No": "",
                    "상품코드": item_code,
                    "구매처분류(중)": buyer_mid,
                    "구매처분류(소)": buyer_small,
                    "구매처분류(세)": buyer_detail,
                    "상품명": product_name,
                    "상품분류(대)": detail_data.get("상품분류(대)", ""),
                    "상품분류(중)": detail_data.get("상품분류(중)", ""),
                    "상품분류(소)": detail_data.get("상품분류(소)", ""),
                    "최소인쇄수량": detail_data.get("최소인쇄수량", ""),
                    "최소주문수량": detail_data.get("최소주문수량", ""),
                    "대량가격(원)": detail_data.get("대량가격(원)", ""),
                    "중간가격(원)": detail_data.get("중간가격(원)", ""),
                    "소량가격(원)": detail_data.get("소량가격(원)", ""),
                    "인쇄방법": row.get("인쇄방법", ""),
                    "업종/행사": row.get("업종/행사", ""),
                    "등록일": row.get("등록일", ""),
                    "수집키": collect_key,
                    "납품사례ID": row.get("납품사례ID", ""),
                    "수집페이지": page,
                    "상세URL": detail_url,
                    "수집상태": "완료",
                }

            except Exception as e:
                err_msg = f"{type(e).__name__}: {e}"
                print(f"[상세 오류] {item_code} / {product_name} / {err_msg}")
                append_error_log(page, item_code, product_name, detail_url, err_msg)

                if DETAIL_ERROR_SAVE_AS_ROW:
                    final_row = make_error_detail_row(row, collect_key, item_code, product_name, detail_url, page)
                else:
                    continue

            checkpoint_df = pd.concat(
                [checkpoint_df, pd.DataFrame([final_row])],
                ignore_index=True
            )

            done_keys.add(collect_key)
            save_checkpoint_files(checkpoint_df)

            total_new_count += 1
            processed_since_excel_save += 1

            print(f"  -> 상세 체크포인트 저장 완료: 누적 {len(checkpoint_df)}건")

            if processed_since_excel_save >= SAVE_EXCEL_EVERY_ROWS:
                save_excel_from_checkpoint(OUTPUT_XLSX)
                processed_since_excel_save = 0

            if total_new_count > 0 and total_new_count % COOLDOWN_EVERY_ITEMS == 0:
                cooldown = random.uniform(COOLDOWN_SLEEP_MIN, COOLDOWN_SLEEP_MAX)
                print(f"[쿨다운] {total_new_count}건 처리 완료, {cooldown:.1f}초 대기")
                time.sleep(cooldown)

    except KeyboardInterrupt:
        print("\n[상세 수집 중지] 현재까지 수집한 데이터 저장 중입니다.")
        save_checkpoint_files(checkpoint_df)
        save_excel_from_checkpoint(OUTPUT_XLSX)
        close_selenium_driver()
        print("[상세 중지 저장 완료] 다음 실행 시 이어서 진행됩니다.")
        return checkpoint_df

    except Exception:
        print("\n[상세 수집 예상 외 오류] 현재까지 수집한 데이터 저장 중입니다.")
        print(traceback.format_exc())
        save_checkpoint_files(checkpoint_df)
        save_excel_from_checkpoint(OUTPUT_XLSX)
        close_selenium_driver()
        print("[상세 오류 저장 완료] 다음 실행 시 이어서 진행됩니다.")
        return checkpoint_df

    save_excel_from_checkpoint(OUTPUT_XLSX)
    close_selenium_driver()

    print(f"\n[2단계 상세 수집 완료] 신규 상세 수집 {total_new_count}건 / 총 저장 {len(checkpoint_df)}건")
    print(f"최종 엑셀: {OUTPUT_XLSX}")

    return checkpoint_df

## 실행 A. 1단계 목록 수집 실행

이 셀을 실행하면 납품사례 목록 페이지를 돌면서 기본 정보와 상세URL을 수집합니다.

실행 결과로 만들어지는 파일은 아래입니다.

- `87sarang_supplycase_list_only.csv`

이 파일에는 상세페이지 주소가 저장됩니다.
2단계 상세 수집은 이 CSV 파일의 상세URL을 기준으로 진행됩니다.

처음 작업할 때는 이 셀을 먼저 실행합니다.
이미 목록 수집 파일이 있으면 중복된 수집키는 건너뛰고 이어서 수집합니다.

주의!!
1개의 page가 끝나야지 저장이 됩니다.


In [ ]:
# =========================
# 1단계 실행: 목록 정보 + 상세링크만 먼저 수집
# =========================

list_df = collect_list_only()

print("목록 수집 건수:", len(list_df))
display(list_df.head(20))

## 실행 B. 2단계 상세 수집 실행

이 셀을 실행하면 `87sarang_supplycase_list_only.csv`에 저장된 상세URL을 하나씩 열어서 상세 정보를 수집합니다.

수집 결과는 중간 저장 파일에 계속 누적됩니다.

- 중간 저장 파일: `87sarang_supplycase_checkpoint.csv`
- 오류 기록 파일: `87sarang_supplycase_error_log.csv`

중간에 멈춰도 체크포인트가 남아 있으면 이어서 실행할 수 있습니다.
이미 수집 완료된 수집키는 다시 수집하지 않습니다.


In [ ]:
# =========================
# 2단계 실행: 저장된 상세링크를 하나씩 열어서 상세정보 수집
# =========================

result_df = run_detail_from_list()

print("최종 체크포인트 건수:", len(result_df))
display(result_df.head(20))

## 실행 C. 수동으로 최종 엑셀 저장하기

이 셀은 현재까지 쌓인 체크포인트 CSV를 기준으로 최종 엑셀을 저장합니다.

상세 수집이 끝난 뒤 실행해도 되고, 중간에 현재까지 수집된 내용을 엑셀로 확인하고 싶을 때 실행해도 됩니다.

저장 파일은 기본 설정의 `OUTPUT_XLSX` 값입니다.
기본 파일명은 `crawl_result.xlsx`입니다. 이 파일에는 `납품사례ID`와 `수집키`도 포함됩니다.


In [ ]:
# =========================
# 수동 저장용 셀
# =========================

save_excel_from_checkpoint(OUTPUT_XLSX)

## 상태 확인. 이어하기와 오류 확인

이 셀은 현재 작업 상태를 확인하는 용도입니다.

확인하는 파일은 아래입니다.

- `87sarang_supplycase_checkpoint.csv`: 상세 수집이 몇 건 저장됐는지 확인합니다.
- `87sarang_supplycase_error_log.csv`: 오류가 몇 건 있었는지 확인합니다.

크롤링이 중간에 멈췄거나 결과가 이상할 때 이 셀을 먼저 실행해서 상태를 보면 됩니다.


In [ ]:
# =========================
# 이어하기 상태 확인
# =========================

if Path(CHECKPOINT_CSV).exists():
    ck = pd.read_csv(CHECKPOINT_CSV, dtype=str)
    print("체크포인트 저장 건수:", len(ck))
    display(ck.tail(10))
else:
    print("체크포인트 파일이 없습니다.")

if Path(ERROR_LOG_CSV).exists():
    err = pd.read_csv(ERROR_LOG_CSV, dtype=str)
    print("오류 로그 건수:", len(err))
    display(err.tail(10))
else:
    print("오류 로그 파일이 없습니다.")